# 0 Install Module & Load Functions

In [ ]:
!pip install pyod
!pip install shap

In [ ]:
from tabulate import tabulate
from scipy.stats import ks_2samp, PermutationMethod

def ks_test(A, B, nameA, nameB):
  tab = []
  for col in range(len(A.columns)):
    stat, pvalue = ks_2samp(A.iloc[:,col], B.iloc[:,col], alternative="two-sided")
    tab.append([A.columns[col], round(pvalue, 3)])

  print(f"KS test on {nameA} vs {nameB}")
  print(tabulate(tab, headers=['Feature', 'p value'], floatfmt=".3f"))


from scipy.stats import anderson_ksamp
def ad_test(A, B, nameA, nameB, method=PermutationMethod()):
  tab = []
  for col in range(len(A.columns)):
    _, _, pval = anderson_ksamp([A.iloc[:,col], B.iloc[:,col]], method=method)
    tab.append([A.columns[col], round(pval, 4)])

  print(f"AD test on {nameA} vs {nameB}")
  print(tabulate(tab, headers=['Feature', 'P Value'], floatfmt=".4f"))

# 1 Anomoly Detection

In [ ]:
# In case ymd are needed
import pandas as pd
df = pd.read_csv("london_weather.csv", parse_dates=["date"])
df = df.set_index("date")
df["year"] = df.index.year
df["month"] = df.index.month
df["day"] = df.index.day

In [ ]:
# Initialize
df["snow_depth"] = df["snow_depth"].fillna(0)
df = df.interpolate(method='linear').iloc[:,:-3]
df = df.ffill().bfill()

# Combine max and min temperature
weather = df.copy()
weather["temp_range"] = weather["max_temp"] - weather["min_temp"]

# Create "snow or not" boolean
snow_mask = weather["snow_depth"] > 0
snow_mask = snow_mask.astype(int)

# Drop the collinear columns
weather = weather.drop(["min_temp", "max_temp", "snow_depth", "global_radiation"], axis=1)

# Deseasonalize by daily
import pandas as pd
Xdeseasoned = pd.DataFrame()
trend = pd.DataFrame()
noise = pd.DataFrame()

from statsmodels.tsa.seasonal import seasonal_decompose
for col in weather.columns:
    decomposition = seasonal_decompose(weather[col], period=365, model='additive', extrapolate_trend='freq')
    Xdeseasoned[col] = decomposition.trend + decomposition.resid
    trend[col] = decomposition.trend
    noise[col] = decomposition.resid

X = Xdeseasoned.ffill().bfill()

# Standardize & Retreive X as DataFrame format
from sklearn.preprocessing import StandardScaler
ss = StandardScaler()
scaled = ss.fit_transform(X.values)
X = pd.DataFrame(scaled, index=X.index, columns=X.columns)

# Replace snow_depth with binary label
X["snow"] = snow_mask

In [ ]:
df.corr()

## 1.0 Correlation

### 1.0.1 Pairplot

In [ ]:
Xpp = X.sample(2000, random_state=42)
Xpp = Xpp.drop("snow", axis=1)
import seaborn as sns
import matplotlib.pyplot as plt

pairplot = sns.pairplot(Xpp, corner=True,
                        plot_kws={"s": 5, "alpha":0.5},
                        diag_kws=dict(fill=False),
                        dropna=True)
pairplot.fig.suptitle("Pairplot on Selected Features", y=1.01, fontsize=16)
pairplot.fig.set_size_inches(6,6)
plt.show()

### 1.0.2 Chatterjee Corr

In [ ]:
from scipy.stats import chatterjeexi
from itertools import combinations
from tabulate import tabulate
import numpy as np

In [ ]:
pairwise = combinations(X.columns, 2)

tab=[]

for i, j in pairwise:
  feat_i = X[i]
  feat_j = X[j]
  corr, pvalue = chatterjeexi(feat_i, feat_j)
  if np.abs(corr) > 0.4:
    tab.append([i, j, round(corr, 3)])
  else:
    continue

print("Chatterjee Correlation larger than 0.4")
print(tabulate(tab, headers=['Feature A', 'Feature B', 'Chatterjee Corr'], floatfmt=".3f"))

## 1.1 Fit Models

In [ ]:
# Fit Iforest
from sklearn.ensemble import IsolationForest
iforest = IsolationForest(
    contamination=0.03,
    n_estimators=150,
    max_samples='auto',
    random_state=42
)
y_if = iforest.fit_predict(X)
y_if = (y_if == -1).astype(int)

X_temp = X.copy()
X_temp['y'] = y_if
X_temp = X_temp[X_temp['y']==1]

df_if = X_temp.drop('y', axis=1)

score_if = iforest.decision_function(X)

# Fit LOF
from pyod.models.lof import LOF
lof = LOF(
    n_neighbors=14,
    contamination=0.03,
    metric='minkowski'
)
lof.fit(X)

y_lof = lof.labels_ == 1
score_lof = lof.predict_proba(X)[:,1]
df_lof = X.loc[y_lof]

# PCA
from sklearn.decomposition import PCA
pca = PCA(n_components=1)
XPCA = pca.fit_transform(X)

print(f"Variance explainable by PCA: {pca.explained_variance_ratio_[0]:.2%}")

# Fit MAD
from pyod.models.mad import MAD
mad = MAD(threshold=2.5)
mad.fit(XPCA)

y_mad = mad.labels_ == 1
df_mad = X.loc[y_mad]

print(f"Length of MAD outliers: {len(df_mad)}")

# Fit knn
from pyod.models.knn import KNN
knn = KNN(contamination=0.03, n_neighbors=14, n_jobs=-1)
knn.fit(X)

y_knn = knn.labels_ == 1
score_knn = knn.decision_function(X)
df_knn = X.loc[y_knn]

import numpy as np
y_z = np.sum(np.abs(X)>2.17, axis=1) >= 2

df_z = X[y_z]

print(f"Length of Z score outliers: {len(df_z)}")

## 1.2 Consensus

In [ ]:
outlier_ind = set()
for df in [df_if, df_lof, df_mad, df_knn, df_z]:
    outlier_ind.update(df.index)

consensus = pd.DataFrame(index=sorted(outlier_ind),
                            columns=['iforest', 'lof', 'mad', 'knn', 'z'])

consensus['iforest'] = consensus.index.isin(df_if.index)
consensus['lof'] = consensus.index.isin(df_lof.index)
consensus['mad'] = consensus.index.isin(df_mad.index)
consensus['knn'] = consensus.index.isin(df_knn.index)
consensus['z'] = consensus.index.isin(df_z.index)

consensus = consensus.astype(int)
consensus['n'] = consensus[['iforest', 'lof', 'mad', 'knn', 'z']].sum(axis=1)

print(f"{len(consensus)} of data ({len(consensus)/len(X):.2%} of the whole data set) are labelled as outliers by any of the algorithms\n")

# Get indices first to prevent overlapping
outlier_ind = set()
for df in [df_if, df_lof, df_knn]:
    outlier_ind.update(df.index)

con3 = pd.DataFrame(index=sorted(outlier_ind),
                            columns=['iforest', 'lof', 'knn'])

con3['iforest'] = con3.index.isin(df_if.index)
con3['lof'] = con3.index.isin(df_lof.index)
con3['knn'] = con3.index.isin(df_knn.index)
con3 = con3.astype(int)

# Set up for both methods = 1
con3['if_lof'] = (con3['iforest'] == con3['lof']) & (con3['iforest'] == 1)
con3['if_knn'] = (con3['iforest'] == con3['knn']) & (con3['iforest'] == 1)
con3['knn_lof'] = (con3['knn'] == con3['lof']) & (con3['knn'] == 1)
con3 = con3.astype(int)
con3['n'] = con3[['iforest', 'lof', 'knn']].sum(axis=1)

from tabulate import tabulate
tab = []
print(f"{len(con3)} of data ({len(con3)/len(X):.2%} of population) are labelled as outliers by any of: [IForest, LOF, kNN]")
print(f"{len(con3[con3['n']==3])} ({len(con3[con3['n']==3])/len(X):.2%} of population) are agreed by all 3 methods\n")

# Set up for either method = 1
if_or_lof = con3['iforest'] + con3['lof'] >= 1
if_or_knn = con3['iforest'] + con3['knn'] >= 1
knn_or_lof = con3['knn'] + con3['lof'] >= 1
agreements = [if_or_lof.sum(), if_or_knn.sum(), knn_or_lof.sum()]

temp = ['if_lof', 'if_knn', 'knn_lof']
for i in temp:
  tab.append([i,
              con3[i].sum(),
              agreements[temp.index(i)],
              f"{con3[i].sum()/agreements[temp.index(i)]:.2%}"])

print(tabulate(tab,
               headers=['Methods', 'Common Count', 'Total Count', 'Proportion of Outliers of the 2 Methods'],
               numalign="center",
               stralign="center"))

# 2 XAI

## 2.0 Permutation Importance

In [ ]:
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt

def feat_plot(model, y1, y2):
  plt.figure(figsize=(6,8))

  plt.subplot(2,1,1)
  model.fit(X, y1)
  res_if = permutation_importance(model, X, y1, n_repeats=10, random_state=42, scoring='accuracy')
  plt.bar(X.columns, res_if.importances_mean)
  plt.xticks(rotation=30)
  plt.title("Feature Importance in Iforest")

  plt.subplot(2,1,2)
  model.fit(X, y2)
  res_lof = permutation_importance(model, X, y2, n_repeats=10, random_state=42, scoring='accuracy')
  plt.bar(X.columns, res_lof.importances_mean)
  plt.xticks(rotation=30)
  plt.title("Feature Importance in LOF")

  plt.tight_layout()
  plt.show()

from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=100,
                            max_depth=5,
                            random_state=42)
feat_plot(rf, y_if, y_lof)

In [ ]:
from sklearn.neural_network import MLPClassifier
mlpc = MLPClassifier(hidden_layer_sizes=(10),
                      max_iter=300,
                      early_stopping=True,
                      random_state=42)
feat_plot(mlpc, y_if, y_lof)

## 2.1 IForest

In [ ]:
# This takes around a minute to load
import shap
import numpy as np

tree_explainer = shap.TreeExplainer(iforest)

shap_if = tree_explainer.shap_values(X)
mashap_if = np.abs(shap_if).mean(axis=0)

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
plt.bar(X.columns, mashap_if)
plt.title('Mean Absolute SHAP Values for IForest')
plt.xticks(rotation=45)

plt.subplot(1,2,2)
shap.summary_plot(shap_if, X, plot_type="dot", max_display=X.shape[1], plot_size=(10,4), show=False)
plt.title("Shap Value Distribution")

plt.tight_layout()
plt.show()

### 2.1.1 Decision Tree

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text

decistree0 = DecisionTreeClassifier(max_depth=3, random_state=42)
decistree0.fit(X, y_if)

plt.figure(figsize=(20, 10))
plot_tree(decistree0,
          feature_names=X.columns,
          class_names=["Normal", "Anomaly"],
          filled=True,
          rounded=True,
          fontsize=18)
plt.title("Decision Tree for IForest Anomalies", fontsize=22)
plt.show()

print("Global Rules for Anomaly Detection:")
print(export_text(decistree0, feature_names=list(X.columns)))

### 2.1.2 Local

In [ ]:
decis_tree_explainer = shap.TreeExplainer(decistree0)
shap_decis = tree_explainer.shap_values(X)
mashap_decis = np.abs(shap_decis).mean(axis=0)

In [ ]:
# Of the 135 un-captured outlier

anom_ind = X.index.get_loc('2020-12-27')
plt.figure(figsize=(6,4))

shap.waterfall_plot(shap.Explanation(values=shap_decis[anom_ind,:],
                    base_values=decis_tree_explainer.expected_value[1],
                    data=X.iloc[anom_ind].values,
                    feature_names=X.columns),
                    show=False)
plt.title(f"Shap Distribution for an Anomalous Day {X.index[anom_ind].date()}")
plt.show()

In [ ]:
# Same node, but the normal class

norm_ind = X.index.get_loc('1979-01-07')

shap.waterfall_plot(shap.Explanation(values=shap_if[norm_ind,:],
                    base_values=decis_tree_explainer.expected_value[0],
                    data=X.iloc[norm_ind].values,
                    feature_names=X.columns),
                    show=False)
plt.title(f"Shap Distribution for a Normal Day {X.index[norm_ind].date()}")
plt.show()

In [ ]:
# Most anomalous day
import matplotlib.pyplot as plt
most_anom_ind = np.argmin(score_if)
shap.waterfall_plot(shap.Explanation(values=shap_decis[most_anom_ind,:],
                    base_values=decis_tree_explainer.expected_value[1],
                    data=X.iloc[most_anom_ind].values,
                    feature_names=X.columns),
                    show=False)
plt.title(f"Shap Distribution for the Most Extreme Day {X.index[most_anom_ind].date()}")
plt.show()

In [ ]:
df = pd.read_csv("london_weather.csv", parse_dates=["date"])
most_snow_ind = np.argmax(df["snow_depth"])
print(df.iloc[most_snow_ind]["date"], df.iloc[most_snow_ind]["snow_depth"])
df.iloc[most_anom_ind]["date"]

** i.e. Most anomalous day has one of the heaviest snow as well, even we have used a binary label instead

We could suspect correlation, meteorologically, such as with mean temperature

### 2.1.3 HP tuning

In [ ]:
from sklearn.ensemble import IsolationForest
from scipy.stats import spearmanr
from tabulate import tabulate
tab = []

mask = pd.DataFrame()

for nest in [100, 400, 600, 800]:
  iforest = IsolationForest(
    contamination=0.02,
    n_estimators=nest,
    max_samples='auto',
    random_state=nest
  )
  iforest.fit(X)
  score_if_hp = iforest.decision_function(X)
  mask[str(nest)] = score_if_hp

mask = mask.astype(float)

for i in [100,400,600,800]:
  corr, p = spearmanr(mask[str(i)], score_if)
  tab.append([i, corr])

print("For Isolation Forest, wrt n_estimator=150:\n")
print(tabulate(tab, headers=['n_estimator', 'Spearman Corr'], floatfmt=".3f"))

## 2.2 LOF

In [ ]:
# Systematic sampling with interval 7, bc it is too slow

import shap
import numpy as np

Xsubsample = X.iloc[::7]
print(f"Size of subsample = {len(Xsubsample)} ({len(Xsubsample)/len(X):.2%})")

kernel_explainer = shap.KernelExplainer(lof.predict_proba, shap.kmeans(Xsubsample, 10))

shap_lof = kernel_explainer.shap_values(Xsubsample, n_jobs=-1)
mashap_lof = np.abs(shap_lof[:,:,1]).mean(axis=0)

In [ ]:
# Validity of Subsample
ks_test(Xsubsample, X, 'Subsample', 'Whole Data')

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
plt.bar(Xsubsample.columns, mashap_lof)
plt.title('Mean Absolute SHAP Values for LOF')
plt.xticks(rotation=45)

plt.subplot(1,2,2)
shap.summary_plot(shap_lof[:,:,1], Xsubsample,
                  plot_type="dot", max_display=Xsubsample.shape[1], plot_size=(10,4), show=False)
plt.title("Shap Value Distribution")

plt.tight_layout()
plt.show()

### 2.2.1 HP tuning

In [ ]:
from pyod.models.lof import LOF
from scipy.stats import spearmanr
from tabulate import tabulate
tab = []

mask = pd.DataFrame()

for nneigh in [7, 20, 25, 30]:
  lof = LOF(
    contamination=0.03,
    n_neighbors=nneigh,
    n_jobs=-1
  )
  lof.fit(X)
  score_lof_hp = lof.predict_proba(X)[:,1]
  mask[str(nneigh)] = score_lof_hp

mask = mask.astype(float)

for i in [7, 20, 25, 30]:
  corr, p = spearmanr(mask[str(i)], score_lof)
  tab.append([i, corr])

print("For LOF, wrt n_neighbors=14:\n")
print(tabulate(tab, headers=['n_neighbors', 'Spearman Corr'], floatfmt=".3f"))

In [ ]:
from pyod.models.lof import LOF
from scipy.stats import spearmanr
from tabulate import tabulate
import numpy as np
tab = []

mask = pd.DataFrame()

for m in ['euclidean', 'manhattan']:
  lof = LOF(
    contamination=0.03,
    n_neighbors=14,
    metric=m,
    n_jobs=-1
  )
  lof.fit(X)
  score_lof_hp = lof.predict_proba(X)[:,1]
  mask[m] = score_lof_hp

mask = mask.astype(float)

for m in ['euclidean', 'manhattan']:
  corr, p = spearmanr(mask[m], score_lof)
  tab.append([m, corr])

print("For LOF, wrt Minkowski:\n")
print(tabulate(tab, headers=['Metrics', 'Spearman Corr'], floatfmt=".3f"))

# 3 Snow-related Issues

## 3.1 Silhouette Score

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

X_snow = X[X["snow"]==1]
X_snow = X_snow.drop("snow", axis=1)

kmeans = KMeans(n_clusters=5, random_state=42, n_init=10).fit(X_snow)
original_score = silhouette_score(X_snow, kmeans.labels_)

tab = []

for i in range(X_snow.shape[1]):
    X_reduced = np.delete(X_snow, i, axis=1)
    kmeans.fit(X_reduced)
    new_score = silhouette_score(X_reduced, kmeans.labels_)
    impact = original_score - new_score
    tab.append([X_snow.columns[i], round(impact,4)])

print("Impact of Features in Terms of Categorizing Snowy Days:\n")
print(tabulate(tab, headers=['Feature', 'Silhouette Score']))

## 3.2 Distribution Tests

In [ ]:
outlier = df_if.copy()
outlier_snow = outlier[outlier["snow"]==1]

X_snow = outlier_snow.drop("snow", axis=1)
X_nosnow = X[X["snow"]==0][::100]

ad_test(X_snow, X_nosnow, 'Snowy Days', 'Non Snowy Days')

## 3.3 If No Snow

In [ ]:
temp = df_if[df_if["snow"]==1]
X_test_if = temp.copy()
X_test_if["snow"] = 0

y_test_if = iforest.predict(X_test_if)
y_test_if = (y_test_if == -1).astype(int)

X_test_if["y"] = y_test_if

print(f"Length of 'Stubborn' Outliers: {len(X_test_if[X_test_if["y"]==1])}")
X_test_if[X_test_if["y"]==1]

### 3.3.1 IForest

In [ ]:
X_331 = X_test_if.drop("y", axis=1)

shap_test = tree_explainer.shap_values(X_331)
mashap_test = np.abs(shap_test).mean(axis=0)

import matplotlib.pyplot as plt
plt.figure(figsize=(6,12))

plt.subplot(2,1,1)
plt.bar(X_331.columns, mashap_test)
plt.title('Mean Absolute SHAP Values for Predicted Values by IForest')
plt.xticks(rotation=45)

plt.subplot(2,1,2)
shap.summary_plot(shap_test, X_331, plot_type="dot", max_display=X_331.shape[1], plot_size=(6,6), show=False)
plt.title("Shap Value Distribution")

plt.tight_layout()
plt.show()

### 3.3.2 LOF

In [ ]:
ind_temp = df_lof[df_lof["snow"]==1].index

Xtemp = X.copy()
Xtemp['snow'] = 0

# We need to refit it bc the original data of the day would severely affect the results
lof.fit(Xtemp)
Xtemp['y'] = lof.labels_

X_ifnosnow = Xtemp.loc[ind_temp]
X_ifnosnow

In [ ]:
len(X_ifnosnow[X_ifnosnow['y']==1])

In [ ]:
X_332 = X_ifnosnow.drop("y", axis=1)

shap_test = kernel_explainer.shap_values(X_332)
mashap_test = np.abs(shap_test[:,:,1]).mean(axis=0)

import matplotlib.pyplot as plt
plt.figure(figsize=(6,12))

plt.subplot(2,1,1)
plt.bar(X_332.columns, mashap_test)
plt.title('Mean Absolute SHAP Values for Predicted Values by LOF')
plt.xticks(rotation=45)

plt.subplot(2,1,2)
shap.summary_plot(shap_test[:,:,1], X_332, plot_type="dot", max_display=X_332.shape[1], plot_size=(6,6), show=False)
plt.title("Shap Value Distribution")

plt.tight_layout()
plt.show()

# 4 Time-related Issues

In [ ]:
anom_if = pd.Series(y_if, index=X.index)
yearly_anom_if = anom_if.resample('YE').sum()

anom_lof = pd.Series(y_lof, index=X.index)
yearly_anom_lof = anom_lof.resample('YE').sum()

import matplotlib.pyplot as plt
plt.figure(figsize=(8, 6))

plt.plot(X.index.year.unique(), yearly_anom_if,
         marker='o', label='iForest')
plt.plot(X.index.year.unique(), yearly_anom_lof,
         marker='s', label='LOF')

plt.title("Yearly Anomaly Frequency (IForest & LOF)")
plt.xlabel("Year")
plt.ylabel("Count")
plt.grid(True, alpha=0.3)
plt.legend()

plt.show()

** Year 2010 & 2020

# 5 Regression

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import RidgeCV, LogisticRegressionCV, LassoCV

def ridge_reg(X, y, degree=2):
  poly = PolynomialFeatures(degree=degree, include_bias=False, interaction_only=True)
  X_poly = poly.fit_transform(X)

  ridge = RidgeCV(cv=5)
  ridge.fit(X_poly, y)

  ridge_df = pd.DataFrame({
    'feature': poly.get_feature_names_out(X.columns),
    'coef': ridge.coef_,
    'abs_coef': np.abs(ridge.coef_)
  })

  return ridge_df.sort_values('abs_coef', ascending=False)

def logis_reg(X, y, degree=2):
  poly = PolynomialFeatures(degree=degree, include_bias=False, interaction_only=True)
  X_poly = poly.fit_transform(X)

  logis = LogisticRegressionCV(random_state=42,
                               solver='saga', max_iter=8000,
                               penalty='l2', n_jobs=-1)
  logis.fit(X_poly, y)

  logis_df = pd.DataFrame({
    'feature': poly.get_feature_names_out(X.columns),
    'coef': logis.coef_[0],
    'abs_coef': np.abs(logis.coef_[0])
  })

  return logis_df.sort_values('abs_coef', ascending=False)

def lasso_reg(X, y, degree=2):
  poly = PolynomialFeatures(degree=degree, include_bias=False, interaction_only=True)
  X_poly = poly.fit_transform(X)

  lasso = LassoCV(cv=5, max_iter=10000)
  lasso.fit(X_poly, y)

  lasso_df = pd.DataFrame({
    'feature': poly.get_feature_names_out(X.columns),
    'coef': lasso.coef_,
    'abs_coef': np.abs(lasso.coef_)
  })

  return lasso_df.sort_values('abs_coef', ascending=False)

** Lasso should best fit our scenerio

## 5.1 Whole Data

In [ ]:
lasso_reg(X, score_if*100, degree=2).head(10)

In [ ]:
logis_reg(X[::7], y_if[::7], degree=2).head(5)

## 5.2 Snowy Days

In [ ]:
Xmerge = X.copy()
Xmerge["score"] = score_if*100

X_snow = Xmerge[Xmerge["snow"]==1]
y_snow = X_snow["score"]
X_snow = X_snow.drop(["snow", "score"], axis=1)

lasso_reg(X_snow, y_snow, degree=2).head(10)